# Feature engineering — Customer Care Calls
Aggregate CC call-level information into per-customer signals.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../../data/02_processed/processed_cc_calls.csv")

df.columns = df.columns.str.lower().str.strip()
df.head()

,contact_id,call_date,direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,co_ref,analysed_call,call_year
0,6.255130e+11,2025-08-05,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30,20,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,NaN,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0,0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,NaN,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,NaN,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,30,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,NaN,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,ED6707,1,2025


In [5]:
df['call_date'] = pd.to_datetime(df['call_date'], errors='coerce')

In [7]:
bill_df = pd.read_csv("../../data/02_processed/processed_billings.csv")
bill_df.columns = bill_df.columns.str.lower().str.strip()

bill_df['prospect_renewal_date'] = pd.to_datetime(
    bill_df['prospect_renewal_date'], errors='coerce'
)

renewal_map = bill_df[['co_ref', 'prospect_renewal_date']].drop_duplicates()

df = df.merge(renewal_map, on='co_ref', how='left')

/var/folders/6q/3_ys7c717hz_scqx9xw7q35w0000gn/T/ipykernel_8101/3797275103.py:1: DtypeWarning: Columns (14,22) have mixed types. Specify dtype option on import or set low_memory=False.
  bill_df = pd.read_csv("../../data/02_processed/processed_billings.csv")


In [8]:
df = df.dropna(subset=['call_date', 'prospect_renewal_date'])

In [9]:
df['cutoff_date'] = df['prospect_renewal_date'] - pd.Timedelta(days=14)

In [10]:
df = df[df['call_date'] <= df['cutoff_date']]

In [11]:
df = df.sort_values(by=['co_ref', 'call_date'])

In [12]:
num_cols = [
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_chasing_response',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_support',
    'cc_contractor_complained',
    'cc_pricing_mentioned',
    'cc_refund_discussed',
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score'
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

In [13]:
agg_df = df.groupby('co_ref').agg({
    'call_date': ['count', 'max']
}).reset_index()

agg_df.columns = ['co_ref', 'total_cc_calls', 'last_cc_call']

In [15]:
cutoff_map = df[['co_ref', 'cutoff_date']].drop_duplicates()

agg_df = agg_df.merge(cutoff_map, on='co_ref', how='left')

agg_df['days_since_last_cc_call'] = (
    agg_df['cutoff_date'] - agg_df['last_cc_call']
).dt.days

In [16]:
df['days_before_cutoff'] = (df['cutoff_date'] - df['call_date']).dt.days

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 7].groupby('co_ref')['call_date'].count().rename('cc_calls_last_7'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 30].groupby('co_ref')['call_date'].count().rename('cc_calls_last_30'),
    on='co_ref', how='left'
)

In [17]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['cc_contractor_complained'].sum().rename('total_complaints'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['cc_customer_issues_concerns'].sum().rename('customer_issues'),
    on='co_ref', how='left'
)

In [18]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['cc_business_struggles_financial_hardship'].sum().rename('financial_issues'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['cc_pricing_mentioned'].sum().rename('pricing_mentions'),
    on='co_ref', how='left'
)

In [19]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['cc_contractor_sentiment_overall_score']
      .mean()
      .rename('avg_cc_sentiment'),
    on='co_ref', how='left'
)

In [20]:
agg_df['repeat_call_ratio'] = (
    agg_df['cc_calls_last_7'] / (agg_df['total_cc_calls'] + 1)
)

agg_df['high_call_volume'] = (agg_df['total_cc_calls'] > 3).astype(int)

In [21]:
agg_df = agg_df.fillna(0)

In [22]:
agg_df.to_csv("../../data/03_final/final_cc_calls_features.csv", index=False)